# The MCP server, from Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adona-Robot/market-data-examples/blob/main/notebooks/mcp.ipynb)

The [adona-robot](https://adona-robot.com/en) MCP server answers plain JSON-RPC over HTTP. Three of its tools need no key: the offer, the symbols, and what a month of usage would cost. The fourth reads candles with your key, billed like the REST API.

An AI agent calls these tools for you (see [the MCP guide](https://adona-robot.com/en/use/market-data-mcp-server-ai-agent)); this notebook calls them by hand, to show what the agent sees.

In [ ]:
%pip install -q requests

In [ ]:
import json

import requests

MCP = "https://api.adona-robot.com/mcp"
HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}


def call(method, params=None, key=None):
    headers = dict(HEADERS, **({"Authorization": f"Bearer {key}"} if key else {}))
    body = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}
    answer = requests.post(MCP, headers=headers, json=body, timeout=30).json()
    return answer["result"]


tools = call("tools/list")["tools"]
[tool["name"] for tool in tools]

## What a month would cost, without a key

In [ ]:
quote = call("tools/call", {"name": "quote_price_for", "arguments": {"simultaneous_connections": 40, "slices_per_month": 100_000}})
print(quote["content"][0]["text"])

In [ ]:
offer = call("tools/call", {"name": "get_offer", "arguments": {}})
print(offer["content"][0]["text"][:1500])

## Candles, with your key

In [ ]:
import os
from getpass import getpass

# The API key, from Colab's secrets (key icon on the left, name ADONA_API_KEY) or typed here.
# It stays in this session: never paste it into a cell's text.
try:
    from google.colab import userdata
    os.environ["ADONA_API_KEY"] = userdata.get("ADONA_API_KEY")
except Exception:
    if not os.environ.get("ADONA_API_KEY"):
        os.environ["ADONA_API_KEY"] = getpass("API key (adk_...): ")

In [ ]:
import os

candles = call(
    "tools/call",
    {"name": "get_candles", "arguments": {"symbol": "XAUUSD", "timeframe": "M1", "limit": 5}},
    key=os.environ["ADONA_API_KEY"],
)
if candles.get("isError"):
    print("refused:", candles["content"][0]["text"])  # the /v1 code, for example CUSTOMER_TRIAL_EXPIRED
else:
    print(candles["content"][0]["text"])

## Further

- Add the server to Claude Code or Cursor: [the configurations](../mcp).
- The same data from Python without MCP: [adona-market-data](https://github.com/Adona-Robot/market-data-python).